# ResFlow quick start

Generate 3D reservoir facies with the pretrained ResFlow model ([huggingface.co/SciLM/ResFlow](https://huggingface.co/SciLM/ResFlow)): one network for eight simulated siliciclastic environments, conditioned on geological parameters and wells, from a single volume to a whole field.

```bash
pip install git+https://github.com/SciLM-ai/ResFlow
```

A GPU makes each step take seconds; a CPU works but is slow, so lower `n` and the field size there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import resflow

model = resflow.load_pretrained()      # downloads the weights once (130 MB)
print(model.environments)

## 1. Volumes from typical parameters

Each volume is 64 × 64 × 32 cells with axes (x, y, z), z = 0 at the base; 1 = sand, 0 = mud. The same `seed` gives the same volumes.

In [ ]:
vols = model.generate('meander', n=8, seed=0)
vols.shape, vols.mean()

In [ ]:
def show(volumes, titles, z=16, cmap='YlOrBr'):
    fig, ax = plt.subplots(1, len(volumes), figsize=(4 * len(volumes), 4))
    for a, v, t in zip(np.atleast_1d(ax), volumes, titles):
        a.imshow(v[:, :, z].T, origin='lower', cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
        a.set_title(t, fontsize=10); a.set_xticks([]); a.set_yticks([])
    plt.show()

show(vols[:4], [f'meander {i}' for i in range(4)])

## 2. All eight environments

In [ ]:
samples = [model.generate(env, n=1, seed=0)[0] for env in model.environments]
show(samples[:4], model.environments[:4]); show(samples[4:], model.environments[4:])

## 3. Choosing parameters

`parameters(env)` lists each parameter as (typical value, training minimum, training maximum). Anything you leave out takes the typical value; widths and depths are in cells, azimuth in degrees (0 = flow along +x). Values outside the training range raise a warning.

In [ ]:
model.parameters('lobe')

In [ ]:
lobes = [model.generate('lobe', n=1, seed=1, ntg=ntg, width_cells=30, azimuth=45)[0] for ntg in (0.2, 0.4, 0.6)]
show(lobes, ['ntg 0.2', 'ntg 0.4', 'ntg 0.6'])

## 4. Conditioning on wells

A `Well` is a vertical column at (x, y) with 1 = sand, 0 = mud and -1 = not observed. Every realisation honours it exactly; the spread between realisations is the model's uncertainty away from the well. For inclined or partial trajectories pass `observed=`, an array of the volume's shape with -1 where nothing is known.

In [ ]:
column = vols[0, 32, 32]                       # a column from a sample, used as a synthetic well
ens = model.generate('meander', n=32, seed=2, wells=[resflow.Well(x=32, y=32, facies=column)])
assert (ens[:, 32, 32] == column).all()
show([ens[0], ens[1], ens.mean(0)], ['realisation 1', 'realisation 2', 'P(sand) over 32'], cmap='viridis')

## 5. Whole fields in one pass

Fields of any size are generated whole, without tiling, by switching to sliding-window attention. Any parameter can also be a 2-D map of shape (X, Y), so trends vary across one reservoir; here lobes shrink along x. Wells and `observed=` work on fields too.

In [ ]:
X, Y = 512, 512
width = np.linspace(60, 30, X)[:, None].repeat(Y, axis=1)
field = model.generate_field('lobe', shape=(X, Y, 32), width_cells=width, ntg=0.5, seed=3)
show([field], ['lobe field: lobes shrink along x'])

In [ ]:
strip = model.generate_field('meander', shape=(512, 128, 32), seed=4)     # flow along x (azimuth 0)
show([strip], ['meander strip, 512 x 128 cells'])